In [21]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


In [2]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [4]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR
Date,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,0.000000
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,0.000000
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,0.000000
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,0.000000
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,0.000000
...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,0.567701
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,0.570294
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,0.553773


In [5]:
resistants_list = Util.define_resistant_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
supports_list = Util.define_support_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

df_h4['support_point'] = df_h4.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: Util.is_support(supports_list, row), axis=1)

df_h4['resistant_point'] = df_h4.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
df_h4['is_resistant'] = df_h4.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4, breakout_candles, key_level_lookback_candle)


In [27]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_h4_resistant_level_list = []
m15_is_resistants_list = []
m15_is_below_resistant_list = []

m15_h4_support_level_list = []
m15_is_supports_list = []
m15_is_above_support_list = []

index_list = []

indicator_atr_m15 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_m15._atr

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_h4_resistant_level_list.append(df_h4.iloc[i]['resistant_point'])
        m15_is_resistants_list.append(df_h4.iloc[i]['is_resistant'])
        m15_is_below_resistant_list.append(df_h4.iloc[i]['valid_breakout_resistant_levels'])
        # affect current market states of H4 candle because the result is market state of previous D1 candle

        m15_h4_support_level_list.append(df_h4.iloc[i]['support_point'])
        m15_is_supports_list.append(df_h4.iloc[i]['is_support'])
        m15_is_above_support_list.append(df_h4.iloc[i]['valid_breakout_support_levels'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['resistant_point'] = m15_h4_resistant_level_list
df_m15['is_resistant'] = m15_is_resistants_list
df_m15['valid_breakout_resistant_levels'] = m15_is_below_resistant_list

df_m15['support_point'] = m15_h4_support_level_list
df_m15['is_support'] = m15_is_supports_list
df_m15['valid_breakout_support_levels'] = m15_is_above_support_list

df_m15.index = index_list

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR,resistant_point,is_resistant,valid_breakout_resistant_levels,support_point,is_support,valid_breakout_support_levels
2021-01-03 23:00:00,140.944,141.110,140.944,141.100,2767,False,True,NaN,0.000000,NaN,False,[207.258],NaN,False,[]
2021-01-03 23:15:00,141.099,141.204,141.070,141.186,1223,False,False,NaN,0.000000,NaN,False,[207.258],NaN,False,[]
2021-01-03 23:30:00,141.186,141.304,141.158,141.282,1042,True,False,NaN,0.000000,NaN,False,[207.258],NaN,False,[]
2021-01-03 23:45:00,141.278,141.301,141.218,141.244,1886,False,False,NaN,0.000000,NaN,False,[207.258],NaN,False,[]
2021-01-04 00:00:00,141.247,141.275,140.988,141.007,3577,False,False,BEARISH,0.000000,NaN,False,[207.258],NaN,False,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-15 23:00:00,208.895,209.029,208.895,209.018,1853,True,False,BEARISH,0.084410,NaN,False,[],NaN,False,[]
2026-09-15 23:15:00,209.020,209.027,208.982,208.996,1268,False,False,BEARISH,0.081595,NaN,False,[],NaN,False,[]
2026-09-15 23:30:00,208.997,209.013,208.926,208.953,1279,False,False,BEARISH,0.081981,NaN,False,[],NaN,False,[]
2026-09-15 23:45:00,208.951,209.021,208.923,209.019,1261,False,False,BEARISH,0.083126,NaN,False,[],NaN,False,[]


In [7]:
def is_existed(list, value):
    for i in range(len(list)):
        if list[i] == value:
            return True
    return False

In [28]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_candle = 150, gap=1, breakout_candles_lookback = 30, start_trade_hour=8, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []
    current_day = None

    # short variables
    is_resistant_level_touched = False
    is_below_resistant_levels = False
    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None
    m15_market_state_bearish = False

    # long variables
    is_support_level_touched = False
    is_above_support_levels = False
    high_tf_condition_long_satisfied = False
    high_tf_condition_long_candle = None
    m15_market_state_bullish = False

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    resistants_levels_touched = []
    supports_levels_touched = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_levels = []
        valid_support_levels = []

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            # short
            high_tf_condition_short_satisfied = False
            resistants_levels_touched = []
            is_resistant_level_touched = False
            valid_breakout_resistant_levels = []
            is_below_resistant_levels = False

            # long
            high_tf_condition_long_satisfied = False
            supports_levels_touched = []
            is_support_level_touched = False
            valid_breakout_support_levels = []
            is_above_support_levels = False

        # short
        c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

        # h4 key level conditions
        resistants_levels = Util.get_resistant_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
        for level in resistants_levels:
            if df['High'].iloc[i-1] > level:
                is_resistant_level_touched = True
                if not is_existed(resistants_levels_touched, level):
                    resistants_levels_touched.append(level)
        
        # Combine conditions
        if (is_resistant_level_touched and
            not high_tf_condition_short_satisfied):
            high_tf_condition_short_satisfied = True
            high_tf_condition_short_candle = current_bar_date
            previous_market_structure = df['market_state'].iloc[i]
            print(f"{current_bar_date} {i}")

        if (high_tf_condition_short_satisfied == True and
            c_m15_ms_bearish and
            high_tf_condition_short_candle < current_bar_date and
            previous_market_structure == 'BULLISH'):
            m15_market_state_bearish = True

        # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
        valid_breakout_resistant_levels = Util.get_valid_breakout_resistant_level(df, i, breakout_candles_lookback)
        
        for valid_level in valid_breakout_resistant_levels:
            for level_touched in resistants_levels_touched:
                if valid_level == level_touched:
                    valid_resistant_levels.append(valid_level)
                    is_below_resistant_levels = True

        c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
        if c_time_trade:
            if (high_tf_condition_short_satisfied == True and
                m15_market_state_bearish and
                is_below_resistant_levels):
                signal = 1

        # long
        c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'
        
        # h4 support level conditions
        supports_levels = Util.get_support_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
        for level in supports_levels:
            if df['Low'].iloc[i-1] < level:
                is_support_level_touched = True
                if not is_existed(supports_levels_touched, level):
                    supports_levels_touched.append(level)

        # Combine conditions
        if (is_support_level_touched and
            not high_tf_condition_long_satisfied):
            high_tf_condition_long_satisfied = True
            m15_market_state_bullish = False
            high_tf_condition_long_candle = current_bar_date
            previous_market_structure = df['market_state'].iloc[i]
            # print(f"high_tf_condition_long_satisfied {current_bar_date} {i}")

        if (high_tf_condition_long_satisfied and
            c_m15_ms_bullish and
            high_tf_condition_long_candle < current_bar_date and
            previous_market_structure == 'BEARISH'):
            # print(f"m15_market_state_bullish {current_bar_date} {i}")
            m15_market_state_bullish = True

        # breakout_candles_above candle needs to be above support level for the breakout to be valid
        valid_breakout_support_levels = Util.get_valid_breakout_support_level(df, i, breakout_candles_lookback)
        for valid_level in valid_breakout_support_levels:
            for level_touched in supports_levels_touched:
                if valid_level == level_touched:
                    valid_support_levels.append(valid_level)
                    is_above_support_levels = True

        c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
        if c_time_trade:
            if (high_tf_condition_long_satisfied == True and
                m15_market_state_bullish and
                is_above_support_levels):
                print(f"signal {current_bar_date} {i}")
                signal = 2

        if previous_market_structure != df['market_state'].iloc[i]:
            if df['market_state'].iloc[i] == 'BULLISH':
                m15_market_state_bearish = False
            if df['market_state'].iloc[i] == 'BEARISH':
                m15_market_state_bullish = False
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_levels)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

df_m15['TotalSignal'], df_m15['valid_resistant_levels'], df_m15['valid_support_levels'] = apply_total_signal(df=df_m15, key_level_lookback_candle=key_level_lookback_candle, gap=gap_look_ahead,
                                                                            breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

2021-01-03 23:00:00 0
2021-04-16 05:45:00 7055
2021-04-16 19:00:00 7108
2021-04-16 19:15:00 7109
2021-04-16 19:30:00 7110
2021-04-16 19:45:00 7111
2021-04-16 20:00:00 7112
2021-04-16 20:15:00 7113
2021-04-16 20:30:00 7114
2021-04-16 20:45:00 7115
2021-04-16 21:00:00 7116
2021-04-16 21:15:00 7117
2021-04-16 21:30:00 7118
2021-04-16 21:45:00 7119
2021-04-19 00:00:00 7120
2021-04-19 19:00:00 7196
2021-04-19 19:15:00 7197
2021-04-19 19:30:00 7198
2021-04-19 19:45:00 7199
2021-04-19 20:00:00 7200
2021-04-19 20:15:00 7201
2021-04-19 20:30:00 7202
2021-04-19 20:45:00 7203
2021-04-19 21:00:00 7204
2021-04-19 21:15:00 7205
2021-04-19 21:30:00 7206
2021-04-19 21:45:00 7207
2021-04-19 22:00:00 7208
2021-04-19 22:15:00 7209
2021-04-19 22:30:00 7210
2021-04-19 22:45:00 7211
2021-04-19 23:00:00 7212
2021-04-19 23:15:00 7213
2021-04-19 23:30:00 7214
2021-04-19 23:45:00 7215
2021-04-20 00:00:00 7216
2021-04-20 19:00:00 7292
2021-04-20 19:15:00 7293
2021-04-20 19:30:00 7294
2021-04-20 19:45:00 7295
202

In [29]:
len(df_m15[df_m15.TotalSignal != 0])

3358

In [30]:
df_m15['pointpos'] = df_m15.apply(lambda row: Util.pointpos(row), axis=1)

In [20]:
start_date_chart = '2023-1-1 00:00'
end_date_chart = '2023-6-30 00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [19]:
start_date_chart = '2023-1-1 00:00'
end_date_chart = '2023-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['resistant_point'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="resistant point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [31]:
class KeyLevelChochStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    entry_gap_multiplier = 0.8
    last_trade_index = 0
    dfopt = None

    stoploss_info_list = []
    valid_resistant_level = None
    valid_support_level = None

    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None
        self.setup_duration = None
        
        self.stoploss_info_list = []
        self.valid_resistant_level = None
        self.valid_support_level = None

        super().init()

    def reset_order(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.order_placed_time = None
        self.setup_duration = None
        self.valid_resistant_level = None
        self.valid_support_level = None

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()

    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window  : len(self.data.Low)].min()

    def is_entry_in_valid_short_zone(self, valid_resistant_list):
        return self.data.Close[-1] < max(valid_resistant_list) + self.data['ATR'][-1] * self.entry_gap_multiplier

    def is_entry_in_valid_long_zone(self, valid_support_list):
        return self.data.Close[-1] > min(valid_support_list) - self.data['ATR'][-1] * self.entry_gap_multiplier

    def add_stoploss_info_list(self, info):
        is_added = False
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == info[0] and stoploss_info[1] == info[1]:
                is_added = True
                stoploss_info[2] = info[2]

        if is_added == False:
            self.stoploss_info_list.append(info)

    def get_stoploss_info(self, key_level_type, key_level_value):
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == key_level_type and stoploss_info[1] == key_level_value:
                return stoploss_info
        return None

    def dynamic_stoploss(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True
                        if order.stop is not None:
                            # in long trade, short order is for sl and inverse for short trade
                            # so we identify short order for long trade to identify sl
                            if order.is_short:
                                self.add_stoploss_info_list([Util.KeyLevel.SUPPORT, self.valid_support_level, order.stop])
                            else:
                                self.add_stoploss_info_list([Util.KeyLevel.RESISTANT, self.valid_resistant_level, order.stop])


                if order_executed == False:
                    order.cancel()
                    if order.is_short:
                        sl = abs(order.stop - self.get_highest_high())
                        tp = order.stop - self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        self.sell(sl = self.get_highest_high(), tp = tp, size=position_size, stop=order.stop)
                    else:
                        sl = abs(order.stop - self.get_lowest_low())
                        tp = order.stop + self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        order = self.buy(sl = self.get_lowest_low(), tp = tp, size=position_size, stop=order.stop)

    def get_tag(self):
        return f"{self.order_placed_time}"

    def wait_3_candles_before_new_trade(self):
        return self.last_trade_index + 3
    
    def next(self):
        super().next()
        index = len(self.data.index)

        # when an order is placed, it will be valable for 12h
        if self.setup_duration is not None and index >= self.setup_duration:
            self.reset_order()

        # place sl to the lowest/highest
        self.dynamic_stoploss()

        if len(self.trades) > 0:
            self.last_trade_index = index

        if len(self.orders)==0 and index >= self.wait_3_candles_before_new_trade():
            if self.data['TotalSignal'][-1]==1:
                valid_resistant_list = Util.get_valid_resistant_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_resistant_list) > 0:
                    valid_resistant_level = max(valid_resistant_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_short_zone(valid_resistant_list)
                    planned_entry_price = 0
                    sl = 0

                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_highest_high())
                    else :
                        planned_entry_price = valid_resistant_level + self.data['ATR'][-1] * self.entry_gap_multiplier
                        sl = abs(planned_entry_price - self.get_highest_high())
    
                    sl1 = planned_entry_price + sl
                    stoploss_info = self.get_stoploss_info(Util.KeyLevel.RESISTANT, valid_resistant_level)
                    if stoploss_info is None or sl1 > stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price - tp

                        self.valid_resistant_level = valid_resistant_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.sell(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & entry_price = {planned_entry_price}")
                        else:
                            self.sell(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & planned_entry_price = {planned_entry_price}")

            if self.data['TotalSignal'][-1]==2:
                valid_support_list = Util.get_valid_support_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_support_list) > 0:
                    valid_support_level = min(valid_support_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_long_zone(valid_support_list)
                    planned_entry_price = 0
                    sl = 0
                    
                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_lowest_low())
                    else :
                        planned_entry_price = valid_support_level - self.data['ATR'][-1] * self.entry_gap_multiplier
                        sl = abs(planned_entry_price - self.get_lowest_low())

                    sl1 = planned_entry_price - sl
                    stoploss_info = self.get_stoploss_info(Util.KeyLevel.SUPPORT, valid_support_level)
                    # print(f"{stoploss_info} {valid_support_level}")
                    if stoploss_info is None or sl1 < stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price + tp

                        self.valid_support_level = valid_support_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.buy(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & entry_price = {planned_entry_price}")
                        else:
                            self.buy(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & planned_entry_price = {planned_entry_price}")
                            

In [33]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_m15.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, KeyLevelChochStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(dfopt=global_dfopt)

print(stats)
bt.plot()


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                     9.54743
Equity Final [$]                   14906.1541
Equity Peak [$]                    14906.1541
Return [%]                           49.06154
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                     6.46823
Volatility (Ann.) [%]                 5.01186
CAGR [%]                              7.25756
Sharpe Ratio                          1.29058
Sortino Ratio                          2.5352
Calmar Ratio                          1.07431
Alpha [%]                            47.53308
Beta                                  0.03173
Max. Drawdown [%]                    -6.02085
Avg. Drawdown [%]                    -0.37078
Max. Drawdown Duration      542 days 08:30:00
Avg. Drawdown Duration        4 days 21:53:00
# Trades                                   68
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1545', ...)

In [40]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 68:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [57]:
trades_list = []
while len(trades_list) != 85:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    10.38285
Equity Final [$]                    11018.183
Equity Peak [$]                     11472.187
Return [%]                           10.18183
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                     1.53404
Volatility (Ann.) [%]                 4.92753
CAGR [%]                              1.71637
Sharpe Ratio                          0.31132
Sortino Ratio                         0.52646
Calmar Ratio                          0.21219
Alpha [%]                            10.35212
Beta                                 -0.00353
Max. Drawdown [%]                    -7.22959
Avg. Drawdown [%]                    -0.51568
Max. Drawdown Duration      490 days 10:30:00
Avg. Drawdown Duration       20 days 07:25:00
# Trades                                   68
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p7793', ...)

In [59]:
i = 0
expectancy_list = []
winrate_list = []
number_trades = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 85:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.0009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 68:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        number_trades.append(stats['# Trades'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

1 1 0.1338511744532744 Done
2 2 -0.027928070341676867 Done
3 3 -0.06421879441695938 Done
4 4 -0.31374650108007845 Done
6 5 -0.14118677160772836 Done
7 6 -0.1672963351495313 Done
8 7 -0.04180133916797766 Done
9 8 0.017526245190933303 Done
10 9 0.09508020974072186 Done
11 10 -0.1567160104736083 Done
13 11 0.05273608493646778 Done
15 12 0.019846029142899367 Done
16 13 -0.0024988751630584866 Done
18 14 0.15380798201866125 Done
20 15 0.0013815197455534305 Done
21 16 -0.1397611250291462 Done
22 17 0.26851214083391706 Done
25 18 -0.004419943622208427 Done
26 19 0.2225647109612682 Done
29 20 -0.03589317865829738 Done
30 21 0.24748613659334043 Done
31 22 -0.12146117240192573 Done
34 23 -0.08691213637097915 Done
35 24 0.22434704413467815 Done
36 25 -0.1415496724398993 Done
38 26 -0.19929909719326472 Done
39 27 0.11121005483295777 Done
40 28 -0.09754298208199821 Done
43 29 0.006925741627377103 Done
44 30 0.13474609936346757 Done
45 31 0.09698091267414045 Done
47 32 -0.049832365328441174 Done
48 3

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23829: Broker canceled the order due to insufficient margin (equity=9987.49, margin_available=9987.49).
  warnings.warn(


91 67 -0.18708311946503148 Done
92 68 0.28056631515031066 Done
93 69 0.01098611617581415 Done
94 70 0.23595513832975692 Done
97 71 0.01591967948126066 Done
98 72 0.03819899959483851 Done
99 73 0.14669545034820444 Done
100 74 -0.05487707762305283 Done
101 75 -0.01793472466748436 Done
102 76 -0.014961357080448481 Done
104 77 -0.045686261572412595 Done
105 78 0.19859117996748935 Done
106 79 -0.09687830619947532 Done
107 80 -0.161145846108919 Done
108 81 -0.14777134566321812 Done
111 82 -0.12785230088361527 Done
112 83 -0.056809407972697276 Done
116 84 0.11400643934069667 Done
117 85 0.1674283216813396 Done
119 86 -0.02750496900052158 Done
120 87 -0.09941370924903352 Done
123 88 0.3289502968762126 Done
124 89 -0.2582697407782755 Done
125 90 0.026708322364953317 Done
127 91 0.034722284958493346 Done
128 92 -0.24884666172515926 Done
129 93 0.00037966571491704023 Done
131 94 -0.05049174521737872 Done
132 95 -0.09411709651972744 Done
133 96 -0.032532825396592725 Done
135 97 0.0716390120053467 

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=27040: Broker canceled the order due to insufficient margin (equity=10295.41, margin_available=10295.41).
  warnings.warn(


139 101 0.05845060813117427 Done
140 102 0.2122023067245553 Done
141 103 0.1270721381267192 Done
143 104 -0.29299316552203797 Done
144 105 -0.0170922334320029 Done
145 106 -0.04397500030527282 Done
147 107 -0.02492880917842398 Done
149 108 -0.06566489786491844 Done
153 109 0.28740280024288534 Done
155 110 0.127955874184884 Done
159 111 0.1733486324235181 Done
161 112 -0.017066111470572538 Done
162 113 0.09092047534737965 Done
163 114 0.08205844359984318 Done
165 115 0.12326326331064724 Done
166 116 0.09955107217595345 Done
167 117 0.14320734756316725 Done
169 118 -0.05824133249889514 Done
170 119 0.12751255466764128 Done
171 120 0.22893502913394675 Done
172 121 -0.11752226522213818 Done
173 122 0.04617940787928765 Done
174 123 -0.10435058743251989 Done
175 124 0.03368950199761486 Done
177 125 0.0831025050715484 Done
178 126 0.11350130242259954 Done
180 127 -0.13100473886347505 Done
181 128 0.12128265477490963 Done
182 129 -0.0900915532882425 Done
186 130 0.17341942434911922 Done
187 13

...

1275 929 -0.10884375039854066 Done
1276 930 0.21575253639129943 Done
1277 931 -0.18431384500620865 Done
1278 932 -0.11318273767352821 Done
1279 933 -0.055485663292014833 Done
1280 934 -0.09091934243811536 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=37547: Broker canceled the order due to insufficient margin (equity=11598.52, margin_available=11598.52).
  warnings.warn(


1281 935 0.20848134732610307 Done
1282 936 -0.1864537417147592 Done
1283 937 -0.2031365453533579 Done
1284 938 -0.060814217525190714 Done
1285 939 0.24878103971839333 Done
1286 940 0.12648080700237366 Done
1287 941 0.4524815154915392 Done
1288 942 0.19311534110810719 Done
1289 943 -0.24457755797451325 Done
1290 944 -0.05542930134369409 Done
1291 945 0.12960720250501329 Done
1293 946 -0.017896045553609664 Done
1295 947 0.2926856208546395 Done
1296 948 -0.09985179156641141 Done
1297 949 0.03186723254456854 Done
1300 950 0.12034151053931666 Done
1301 951 -0.011038165036240371 Done
1302 952 0.1640117487728351 Done
1303 953 0.08434194499520761 Done
1304 954 0.14881602727076865 Done
1305 955 0.003326351432163663 Done
1307 956 0.09177471607120301 Done
1308 957 -0.13655458902564918 Done
1309 958 0.06553202722761314 Done
1310 959 0.010285579436758126 Done
1313 960 0.1289908908753688 Done
1314 961 0.09263188670040857 Done
1315 962 -0.22900969066000187 Done
1317 963 0.11063771755595748 Done
1318 

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=104579: Broker canceled the order due to insufficient margin (equity=9651.30, margin_available=9651.30).
  warnings.warn(


1349 984 0.007270710295339936 Done
1350 985 -0.12535093347231127 Done
1352 986 -0.09135749584769566 Done
1353 987 0.0014194515263696143 Done
1355 988 0.3163481687403367 Done
1356 989 0.034569215530182405 Done
1358 990 0.09507446512611872 Done
1359 991 0.08001604059942667 Done
1360 992 -0.15238567700901304 Done
1361 993 0.09924907841860701 Done
1362 994 0.1936668415147819 Done
1363 995 -0.07989444255040952 Done
1364 996 -0.01998574309835799 Done
1366 997 0.07842327874614693 Done
1367 998 0.0029068905336532606 Done
1369 999 -0.03569162973856533 Done
1371 1000 -0.09511022829354408 Done
[np.float64(0.1338511744532744), np.float64(-0.027928070341676867), np.float64(-0.06421879441695938), np.float64(-0.31374650108007845), np.float64(-0.14118677160772836), np.float64(-0.1672963351495313), np.float64(-0.04180133916797766), np.float64(0.017526245190933303), np.float64(0.09508020974072186), np.float64(-0.1567160104736083), np.float64(0.05273608493646778), np.float64(0.019846029142899367), np.flo

In [60]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {0.32898 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= 0.32898).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= 0.32898).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {54.41176 - mean_winrate}")

1000
p95 = 0.24333519402902032
max = 0.4929891563669567
mean = 0.021174371872601447
min = -0.31374650108007845
gap = 0.30780562812739853
rank = 99.3
p_value = 0.007992007992007992
1000
p95 = 45.588235294117645
max = 54.41176470588235
mean = 35.05882352941176
min = 20.588235294117645
gap = 19.35293647058824
